In [297]:
import numpy as np
import pandas as pd

In [298]:
from surprise import SVD, KNNBaseline, Dataset, Reader
from surprise.model_selection import train_test_split
from surprise import accuracy
from surprise.model_selection import cross_validate

In [299]:
import time

In [300]:
path = "interactions_data/"

In [301]:
df_movies = pd.read_csv(path + 'movies_data.csv')
df_interactions = pd.read_csv(path + 'interactions_data.csv')

In [302]:
df_movies.rename(columns={'id': 'movie_id', 'title': 'title', 'genres': 'genres'}, inplace=True)

In [303]:
df_movies.head()

,movie_id,title,description,genres,actors,directors
0,37702,Forbidden City Cop,An imperial agent gets ridiculed for his vario...,"Adventure, Action, Comedy","Stephen Chow, Carina Lau, Carman Lee Yeuk-Tung...",Vincent Kok Tak-Chiu
1,1290821,Shelter,A man living in self-imposed exile on a remote...,"Action, Thriller, Crime","Jason Statham, Bodhi Rae Breathnach, Bill Nigh...",Ric Roman Waugh
2,1171145,Crime 101,When an elusive thief whose high-stakes heists...,"Thriller, Crime","Chris Hemsworth, Mark Ruffalo, Halle Berry, Ba...",Bart Layton
3,1198994,Send Help,Two colleagues become stranded on a deserted i...,"Horror, Comedy, Thriller","Rachel McAdams, Dylan O'Brien, Edyll Ismail, D...",Sam Raimi
4,840464,Greenland 2: Migration,Having found the safety of the Greenland bunke...,"Adventure, Thriller, Science Fiction","Gerard Butler, Morena Baccarin, Roman Griffin ...",Ric Roman Waugh


In [304]:
df_interactions.head()

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748


In [305]:
def calculate_base_score(row, weights):
    v_progress = row['max_progress_percent'] / 100
    v_rating = row['rating'] / 5.0 if pd.notnull(row['rating']) else None
    v_fav = float(row['favourite'])
    v_watchlist = float(row['in_watchlist'])
    v_view_count = np.log(max(row['view_count'], 1)) / np.log(10)

    # Tính điểm theo từng thành phần hợp lệ
    numerator = (
        weights['progress'] * v_progress
        + weights['favourite'] * v_fav
        + weights['watchlist'] * v_watchlist
        + weights['view_count'] * v_view_count
    )
    denominator = (
        weights['progress']
        + weights['favourite']
        + weights['watchlist']
        + weights['view_count']
    )

    # Chỉ cộng rating khi có dữ liệu
    if v_rating is not None:
        numerator += weights['rating'] * v_rating
        denominator += weights['rating']

    return numerator / denominator * 10

In [306]:
weights = {
    'progress': 3.0,
    'rating': 4.1,
    'favourite': 4.5,
    'watchlist': 2.0,
    'view_count': 1.1
}

df_interactions['base_score'] = df_interactions.apply(calculate_base_score, axis=1, weights=weights)

In [307]:
df_interactions

,user_id,movie_id,rating,max_progress_percent,favourite,in_watchlist,view_count,last_watched_at,base_score
0,10,69492,NaN,85,0,1,3,2025-12-18 01:56:05.017054,4.787579
1,10,28932,NaN,80,0,0,1,2026-02-02 01:56:05.032371,2.264151
2,10,372754,NaN,85,0,1,2,2025-12-10 01:56:05.042977,4.604842
3,10,210479,NaN,82,0,0,6,2025-10-20 01:56:05.053147,3.128270
4,10,16071,NaN,80,0,0,7,2025-10-30 01:56:05.064748,3.141139
...,...,...,...,...,...,...,...,...,...
27011,500,15276,5.0,97,1,0,4,2025-12-06 01:59:01.806758,8.280453
27012,500,736545,5.0,81,1,1,3,2026-03-08 01:59:01.818289,9.220975
27013,500,11297,4.0,92,0,0,2,2025-10-01 01:59:01.826666,4.334104
27014,500,146724,5.0,90,1,0,3,2025-10-04 01:59:01.838193,8.044104


In [308]:
from collections import Counter

In [309]:
# Algorithm 1: Phân loại thể loại yêu thích và không thích của người dùng

def get_user_genres_preferences(user_id, df_interaction, df_items):
  # Get movieId which user rated
  user_ratings = df_interactions[df_interaction['user_id'] == user_id]

  # Count genre
  genre_counts = Counter()
  for m_id in user_ratings['movie_id']:
    # Get string genres of movie ("Action|Adventure")
    genre_str = df_items.loc[df_items['movie_id'] == m_id, 'genres'].values[0]
    if pd.isna(genre_str): continue
    for genre in genre_str.split(', '):
      genre_counts[genre] += 1

  # Get all genre
  all_genres = set()
  df_items['genres'].str.split(', ').apply(lambda x: all_genres.update(x) if isinstance(x, list) else None)

  # Sort asc
  sorted_genres = sorted(list(all_genres), key=lambda x: genre_counts.get(x, 0), reverse=True)

  # Find mid-point
  mid_point = len(sorted_genres) // 2
  fav_genres = sorted_genres[:mid_point]
  unfav_genres = sorted_genres[mid_point:]

  return fav_genres, unfav_genres


In [310]:
# Algorithm 2: Kiểm tra tính liên quan (Relevance) dựa trên Genre
def is_movie_relevant(movie_id, df_movies, fav_genres, unfav_genres):
    res = df_movies.loc[df_movies['movie_id'] == movie_id, 'genres'].values
    if len(res) == 0 or pd.isna(res[0]): return False

    movie_genres = [g.strip() for g in res[0].split(', ')]
    # Score = (Số genre thích) - (Số genre ghét)
    score = sum(1 for g in movie_genres if g in fav_genres) - \
            sum(1 for g in movie_genres if g in unfav_genres)
    return score > 0

In [317]:
# Algorithm 4: Tính các chỉ số Precision@K, Recall@K, F1 score
def calculate_metrics_at_k(predictions, df_interaction, df_movies, k):
  user_est_true = {}
  for uid, iid, true_r, est, _ in predictions:
    if uid not in user_est_true:
      user_est_true[uid] = []
    user_est_true[uid].append((iid, est))

  precisions = []
  recalls = []

  for uid, user_ratings in user_est_true.items():
    # Lấy gu của user
    fav, unfav= get_user_genres_preferences(uid, df_interaction, df_movies)

    # Sắp xếp dự đoán theo điểm 'est' cao nhất
    user_ratings.sort(key=lambda x: x[1], reverse=True)
    top_k = user_ratings[:k]

    # Tính TP (True Positives)
    n_rel_and_rec_k = sum(1 for (iid, est) in top_k if is_movie_relevant(iid, df_movies, fav, unfav))

    # Tính tổng số phim relevant thực tế có trong tập test của user
    n_rel = sum(1 for (iid, est) in user_ratings if is_movie_relevant(iid, df_movies, fav, unfav))

    # Tính precision and recall cho user
    precisions.append(n_rel_and_rec_k / k)
    recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

  # Lấy avg
  avg_prec = np.mean(precisions)
  avg_rec = np.mean(recalls)

  f1 = (2 * avg_prec * avg_rec) / (avg_rec + avg_prec) if (avg_rec + avg_prec) >0 else 0

  return avg_prec, avg_rec, f1

In [318]:
from surprise import SVD, KNNBaseline, CoClustering, NMF, SlopeOne, BaselineOnly

In [319]:
# Danh sách các thuật toán
algorithms = [
    SVD(),
    CoClustering(),
    NMF(),
    SlopeOne(),
    KNNBaseline(),
    BaselineOnly()
]

In [320]:
# Chia dữ liệu Trainning và test, tỉ lệ 75/25
reader = Reader(rating_scale=(0, 10))

data = Dataset.load_from_df(df_interactions[['user_id', 'movie_id', 'base_score']], reader)

trainset, testset = train_test_split(data, test_size=0.25)

In [321]:
# Huấn luyện, dự đoán và đo thời gian
results_table = []

for algorithm in algorithms:
  algo_name = algorithm.__class__.__name__

  start_time = time.time()

  # Train
  algorithm.fit(trainset)

  # Predict
  predictions = algorithm.test(testset)

  #Time
  eval_time = time.time() - start_time


  # Tinh RMSE va MAE
  rmse_val = accuracy.rmse(predictions, verbose=False)
  mae_val = accuracy.mae(predictions, verbose=False)

  # Tinh Precision, recall, f1
  prec, rec, f1 = calculate_metrics_at_k(predictions, df_interactions, df_movies, k = 15)

  results_table.append([algo_name, rmse_val, mae_val, prec, rec, f1, eval_time])
  print(f"{algo_name:<15} | {rmse_val:<8.4f} | {mae_val:<8.4f} | {prec:<10.4f} | {rec:<8.4f} | {f1:<8.4f} | {eval_time:<8.4f}")


SVD             | 1.9575   | 1.4692   | 0.7344     | 0.9426   | 0.8256   | 0.4175  
CoClustering    | 2.6208   | 2.0696   | 0.7347     | 0.9429   | 0.8259   | 0.9982  
NMF             | 2.4912   | 1.9173   | 0.7359     | 0.9440   | 0.8271   | 0.7932  
SlopeOne        | 2.2904   | 1.7189   | 0.7360     | 0.9442   | 0.8272   | 0.9236  
Estimating biases using als...
Computing the msd similarity matrix...
Done computing similarity matrix.
KNNBaseline     | 2.1613   | 1.6407   | 0.7351     | 0.9432   | 0.8262   | 0.0963  
Estimating biases using als...
BaselineOnly    | 1.9753   | 1.4986   | 0.7337     | 0.9419   | 0.8249   | 0.0546  


In [323]:

# Xuất ra DataFrame 
df_final_results = pd.DataFrame(results_table, columns=['Algorithms', 'RMSE', 'MAE', 'Precision', 'Recall', 'F1 Score', 'Time'])
df_final_results

,Algorithms,RMSE,MAE,Precision,Recall,F1 Score,Time
0,SVD,1.957547,1.469183,0.734420,0.942620,0.825596,0.417533
1,CoClustering,2.620812,2.069606,0.734691,0.942918,0.825882,0.998192
2,NMF,2.491200,1.917316,0.735913,0.943985,0.827063,0.793185
3,SlopeOne,2.290428,1.718929,0.736049,0.944193,0.827229,0.923576
4,KNNBaseline,2.161273,1.640750,0.735098,0.943185,0.826242,0.096289
5,BaselineOnly,1.975302,1.498595,0.733741,0.941902,0.824892,0.054614


In [324]:
print(predictions)

[Prediction(uid=181, iid=139582, r_ui=7.496331866677336, est=5.209874434956224, details={'was_impossible': False}), Prediction(uid=459, iid=194662, r_ui=8.878321765462843, est=5.399477461631739, details={'was_impossible': False}), Prediction(uid=445, iid=515916, r_ui=1.3867924528301887, est=2.1600961836508907, details={'was_impossible': False}), Prediction(uid=303, iid=421, r_ui=8.961495908633427, est=5.847541362440747, details={'was_impossible': False}), Prediction(uid=92, iid=504562, r_ui=3.187610381858133, est=2.8989962517067185, details={'was_impossible': False}), Prediction(uid=268, iid=554776, r_ui=3.4560367789539037, est=3.2838110742508277, details={'was_impossible': False}), Prediction(uid=106, iid=1248226, r_ui=2.9172320664724136, est=3.0142310600734867, details={'was_impossible': False}), Prediction(uid=423, iid=45657, r_ui=4.717573728021517, est=6.184191893550116, details={'was_impossible': False}), Prediction(uid=131, iid=792237, r_ui=2.1415094339622645, est=2.3954521140943

In [325]:

def get_recommendations(user_id, model, df_interactions, df_movies, top_k=15):
    # 1. Lấy danh sách tất cả movie_id trong hệ thống
    all_movie_ids = df_movies['movie_id'].unique()
    
    # 2. Tìm các phim user này đã xem để loại bỏ
    watched_movies = df_interactions[df_interactions['user_id'] == user_id]['movie_id'].unique()
    
    # 3. Lọc ra danh sách các phim user CHƯA xem
    unseen_movies = [m_id for m_id in all_movie_ids if m_id not in watched_movies]
    
    # 4. Dự đoán điểm cho từng bộ phim chưa xem
    predictions = []
    for m_id in unseen_movies:
        # model.predict(uid, iid) trả về một object có thuộc tính 'est' là điểm dự đoán
        pred = model.predict(user_id, m_id)
        predictions.append((m_id, pred.est))
    
    # 5. Sắp xếp theo điểm dự đoán giảm dần
    predictions.sort(key=lambda x: x[1], reverse=True)
    
    # 6. Lấy Top K phim đầu bảng
    top_recommendations = predictions[:top_k]
    
    result = []
    for m_id, score in top_recommendations:
        movie_info = df_movies[df_movies['movie_id'] == m_id].iloc[0]
        result.append({
            'movie_id': m_id,
            'title': movie_info['title'],
            'genres': movie_info['genres'],
            'predicted_score': round(score, 2)
        })
        
    return pd.DataFrame(result)

user_test = 42 # Thử với user bất kỳ
recommendations = get_recommendations(user_test, algo, df_interactions, df_movies, top_k=20)
print(f"Top 20 phim gợi ý cho User {user_test}:")
print(recommendations)

Top 20 phim gợi ý cho User 42:
    movie_id                                            title  \
0      10206                         Buffy the Vampire Slayer   
1     467905                            How to Make a Killing   
2      10925                    The Return of the Living Dead   
3       8090                                      Untraceable   
4       4552                            A Tale of Two Sisters   
5       2265                                             Momo   
6      75983                            The Rules of the Game   
7      10521                                       Bride Wars   
8     194662  Birdman or (The Unexpected Virtue of Ignorance)   
9      10951                                         Gorgeous   
10   1302916                                       Heart Eyes   
11    537971                                              Mia   
12     78139             Lupin the Third: Alcatraz Connection   
13     28032                                       Thumbeli

In [328]:
fav_genres, unfav_genres = get_user_genres_preferences(user_test, df_interactions, df_movies)
print(f"User {user_test} thích thể loại: {fav_genres}")
print(f"User {user_test} không thích thể loại: {unfav_genres}")

User 42 thích thể loại: ['Action', 'Thriller', 'Drama', 'Adventure', 'Comedy', 'Crime', 'Family', 'Animation', 'Horror']
User 42 không thích thể loại: ['Fantasy', 'Romance', 'Science Fiction', 'Mystery', 'War', 'History', 'Western', 'TV Movie', 'Music', 'Documentary']


In [332]:
count = 0
for row in recommendations.itertuples():
    if is_movie_relevant(row.movie_id, df_movies, fav_genres, unfav_genres):
        count += 1
print(f"Tổng số phim phù hợp với sở thích của User {user_test}: {count} / {len(recommendations)}")

Tổng số phim phù hợp với sở thích của User 42: 15 / 20
